# gradcam.ipynb — Interprétabilité Grad-CAM
Visualise les zones de la radiographie influençant la décision du CNN.

> **Extension optionnelle — Jour J9**

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, '.')

import cv2
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms

%run model.ipynb

CHECKPOINT = '../outputs/checkpoints/best_model.pt'
IMAGE_SIZE_GC = 224
MEAN_GC = (0.485, 0.456, 0.406)
STD_GC  = (0.229, 0.224, 0.225)
FIG_DIR = Path('../outputs/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Grad-CAM pret sur {DEVICE.upper()}')

## Classe GradCAM

In [ ]:
class GradCAM:
    """Grad-CAM par hooks forward/backward sur la couche cible."""
    def __init__(self, model, target_layer, device='cpu'):
        self.model  = model.to(device).eval()
        self.device = device
        self._grads = None
        self._acts  = None
        target_layer.register_forward_hook(lambda m, inp, out: setattr(self, '_acts', out.detach()))
        target_layer.register_full_backward_hook(lambda m, gi, go: setattr(self, '_grads', go[0].detach()))

    def generate(self, img_tensor):
        img_tensor = img_tensor.to(self.device).requires_grad_(True)
        out = self.model(img_tensor)
        self.model.zero_grad()
        out.backward()
        weights = self._grads.mean(dim=(2, 3), keepdim=True)   # (1,C,1,1)
        cam = torch.relu((weights * self._acts).sum(dim=1, keepdim=True))
        cam = cam.squeeze().cpu().numpy()
        return cam / cam.max() if cam.max() > 0 else cam

print('Classe GradCAM definie.')

## Chargement image + modèle

In [ ]:
def load_image(path):
    tf = transforms.Compose([
        transforms.Resize((IMAGE_SIZE_GC, IMAGE_SIZE_GC)),
        transforms.ToTensor(),
        transforms.Normalize(mean=MEAN_GC, std=STD_GC),
    ])
    img_pil = Image.open(path).convert('RGB')
    return tf(img_pil).unsqueeze(0), np.array(img_pil.resize((IMAGE_SIZE_GC, IMAGE_SIZE_GC)))

# Chargement du modèle
ckpt  = torch.load(CHECKPOINT, map_location=DEVICE)
model_gc = get_model(ckpt.get('architecture', 'baseline'))
model_gc.load_state_dict(ckpt['model_state_dict'])

# Couche cible : dernière ReLU du dernier ConvBlock
target_layer = model_gc.features[-1].block[-2]
gradcam = GradCAM(model_gc, target_layer, device=DEVICE)
print('Modele et GradCAM prets.')

## Génération de la heatmap
Modifiez `IMAGE_PATH` pour pointer vers l'image souhaitée.

In [ ]:
# Choisir une image du test set
IMAGE_PATH = list(Path('../data/chest_xray/test/PNEUMONIA').glob('*.jpeg'))[0]
print(f'Image : {IMAGE_PATH.name}')

img_tensor, img_np = load_image(IMAGE_PATH)
prob = torch.sigmoid(model_gc(img_tensor)).item()
pred = 'PNEUMONIA' if prob >= 0.5 else 'NORMAL'

# CAM
cam = gradcam.generate(img_tensor)
cam_resized = cv2.resize(cam, (img_np.shape[1], img_np.shape[0]))
heatmap = cv2.cvtColor(cv2.applyColorMap((cam_resized*255).astype(np.uint8), cv2.COLORMAP_JET), cv2.COLOR_BGR2RGB)
superimposed = np.clip(cv2.addWeighted(img_np, 0.5, heatmap, 0.5, 0), 0, 255).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img_np);        axes[0].set_title('Originale');   axes[0].axis('off')
axes[1].imshow(cam, cmap='jet'); axes[1].set_title('CAM');       axes[1].axis('off')
axes[2].imshow(superimposed);  axes[2].set_title(f'Superposee\n{pred} ({prob:.2f})'); axes[2].axis('off')

plt.suptitle(f'Grad-CAM — {IMAGE_PATH.name}', fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'gradcam.png', dpi=150)
%matplotlib inline
plt.show()
print('Saved -> outputs/figures/gradcam.png')